In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from datetime import timedelta

In [ ]:
df = pd.read_csv('/content/return_log.csv')

# Convert the 'date' column to datetime
df['date'] = pd.to_datetime(df['date'])

# Set the date as the index of the DataFrame
df.set_index('date', inplace=True)

# Check the DataFrame structure
print(df.head())

In [ ]:
#df_clean = df.fillna(method='ffill')
#df_clean = df.fillna(0)
df_clean = df

print(df_clean.head())

In [ ]:
event_df = pd.read_csv('/content/only_weak_experience_mess.csv')
#event_df = pd.read_csv('/content/event date mess.csv')
# Convert 'event_date' to datetime
event_df['event_date'] = pd.to_datetime(event_df['event_date'], format='%d/%m/%Y')

# Check the structure of the event data
print(event_df.head())

In [ ]:
from datetime import timedelta

# Define window lengths
event_window_length = 2  # +/- 7 days for the event window (counting dataset dates, not calendar dates)
lag_period = 14  # 14 trading days before the event window
estimation_window_length = 180  # 90 trading days for the estimation window

# Function to get estimation, lag, and event windows based on available dates in the dataset
def get_windows(data, event_date, estimation_length, lag_period, event_length):
    # Get the index position of the event date
    event_idx = data.index.get_loc(event_date)

    # Define window ranges using index positions
    estimation_end_idx = event_idx - lag_period - event_length
    estimation_start_idx = estimation_end_idx - estimation_length

    event_start_idx = event_idx - event_length
    event_end_idx = event_idx + event_length

    # Slice the data based on index positions, handling non-trading days
    estimation_window = data.iloc[estimation_start_idx:estimation_end_idx]
    event_window = data.iloc[event_start_idx:event_end_idx + 1]  # Include the event day

    return estimation_window, event_window


In [ ]:
# Скипает даты
import statsmodels.api as sm

# Initialize an empty list to store the results
results = []

# Loop through each event
for index, row in event_df.iterrows():
    company_ticker = row['security_ticker']
    event_date = row['event_date']

    # Ensure event_date exists in the dataset, skip if it doesn't
    if event_date not in df_clean.index:
        print(f"Event date {event_date} for {company_ticker} not found in dataset. Skipping...")
        continue

    # Extract stock and market returns
    company_returns = df_clean[company_ticker]
    market_returns = df_clean['MOEX']

    # Get the windows (estimation, event)
    estimation_window, event_window = get_windows(df_clean, event_date, estimation_window_length, lag_period, event_window_length)

    # Perform regression in the estimation window (Market Model)
    X = sm.add_constant(estimation_window['MOEX'])
    model = sm.OLS(estimation_window[company_ticker], X).fit()

    # Get alpha and beta values
    alpha, beta = model.params

    # Calculate predicted returns and abnormal returns (AR) during the event window
    predicted_returns = alpha + beta * event_window['MOEX']
    abnormal_returns = event_window[company_ticker] - predicted_returns

    # Calculate cumulative abnormal returns (CAR)
    cumulative_abnormal_returns = abnormal_returns.cumsum()

    # Store the results
    for date, ar, car in zip(event_window.index, abnormal_returns, cumulative_abnormal_returns):
        results.append({
            'security_ticker': company_ticker,
            'event_date': event_date,
            'date': date,
            'AR': ar,
            'CAR': car
        })

# Convert the results to a DataFrame
results_df = pd.DataFrame(results)

# Check the first few rows of the results
print(results_df.head())

In [ ]:
# Использует след доступ дату
import statsmodels.api as sm

# Initialize an empty list to store the results
results = []

# Loop through each event
for index, row in event_df.iterrows():
    company_ticker = row['security_ticker']
    event_date = row['event_date']

    # Ensure event_date exists in the dataset, find the next available date if it doesn't
    if event_date not in df_clean.index:
        # Find the next available date
        next_available_date = df_clean.index[df_clean.index > event_date].min()
        if pd.isna(next_available_date):
            print(f"No future date found in dataset for event date {event_date} of {company_ticker}. Skipping...")
            continue
        print(f"Event date {event_date} for {company_ticker} not found. Using next available date: {next_available_date}")
        event_date = next_available_date

    # Extract stock and market returns
    company_returns = df_clean[company_ticker]
    market_returns = df_clean['MOEX']

    # Get the windows (estimation, event)
    estimation_window, event_window = get_windows(df_clean, event_date, estimation_window_length, lag_period, event_window_length)

    # Perform regression in the estimation window (Market Model)
    X = sm.add_constant(estimation_window['MOEX'])
    model = sm.OLS(estimation_window[company_ticker], X).fit()

    # Get alpha and beta values
    alpha, beta = model.params

    # Calculate predicted returns and abnormal returns (AR) during the event window
    predicted_returns = alpha + beta * event_window['MOEX']
    abnormal_returns = event_window[company_ticker] - predicted_returns

    # Calculate cumulative abnormal returns (CAR)
    cumulative_abnormal_returns = abnormal_returns.cumsum()

    # Store the results
    for date, ar, car in zip(event_window.index, abnormal_returns, cumulative_abnormal_returns):
        results.append({
            'security_ticker': company_ticker,
            'original_event_date': row['event_date'],
            'adjusted_event_date': event_date,
            'date': date,
            'AR': ar,
            'CAR': car
        })

# Convert the results to a DataFrame
results_df = pd.DataFrame(results)

# Check the first few rows of the results
print(results_df.head())

In [ ]:
# считает CAR
import statsmodels.api as sm

# Initialize an empty list to store the results
results = []

# Loop through each event
for index, row in event_df.iterrows():
    company_ticker = row['security_ticker']
    event_date = row['event_date']

    # Ensure event_date exists in the dataset, skip if it doesn't
    if event_date not in df_clean.index:
        print(f"Event date {event_date} for {company_ticker} not found in dataset. Skipping...")
        continue

    # Extract stock and market returns
    company_returns = df_clean[company_ticker]
    market_returns = df_clean['MOEX']

    # Get the windows (estimation, event)
    estimation_window, event_window = get_windows(df_clean, event_date, estimation_window_length, lag_period, event_window_length)

    # Perform regression in the estimation window (Market Model)
    X = sm.add_constant(estimation_window['MOEX'])
    model = sm.OLS(estimation_window[company_ticker], X).fit()

    # Get alpha and beta values
    alpha, beta = model.params

    # Calculate predicted returns and abnormal returns (AR) during the event window
    predicted_returns = alpha + beta * event_window['MOEX']
    abnormal_returns = event_window[company_ticker] - predicted_returns

    # Calculate cumulative abnormal returns (CAR)
    cumulative_abnormal_returns = abnormal_returns.cumsum()

     # Store the CAR for the last day of the event window
    last_date = event_window.index[-1]
    last_car = cumulative_abnormal_returns.iloc[-1]

    results.append({
                'security_ticker': company_ticker,
                'event_date': event_date,
                'date': last_date,
                'CAR': last_car
            })

# Convert the final results to a DataFrame
results_df = pd.DataFrame(results)

# Check the first few rows of the results
print(results_df.head())

In [ ]:
# CAR с послед доступ датой
import statsmodels.api as sm

# Initialize an empty list to store the results
results = []

# Loop through each event
for index, row in event_df.iterrows():
    company_ticker = row['security_ticker']
    event_date = row['event_date']

    # Ensure event_date exists in the dataset, find the next available date if it doesn't
    if event_date not in df_clean.index:
        # Find the next available date
        next_available_date = df_clean.index[df_clean.index > event_date].min()
        if pd.isna(next_available_date):
            print(f"No future date found in dataset for event date {event_date} of {company_ticker}. Skipping...")
            continue
        print(f"Event date {event_date} for {company_ticker} not found. Using next available date: {next_available_date}")
        event_date = next_available_date

    # Extract stock and market returns
    company_returns = df_clean[company_ticker]
    market_returns = df_clean['MOEX']

    # Get the windows (estimation, event)
    estimation_window, event_window = get_windows(df_clean, event_date, estimation_window_length, lag_period, event_window_length)

    # Perform regression in the estimation window (Market Model)
    X = sm.add_constant(estimation_window['MOEX'])
    model = sm.OLS(estimation_window[company_ticker], X).fit()

    # Get alpha and beta values
    alpha, beta = model.params

    # Calculate predicted returns and abnormal returns (AR) during the event window
    predicted_returns = alpha + beta * event_window['MOEX']
    abnormal_returns = event_window[company_ticker] - predicted_returns

    # Calculate cumulative abnormal returns (CAR)
    cumulative_abnormal_returns = abnormal_returns.cumsum()

    # Store the CAR for the last day of the event window
    last_date = event_window.index[-1]
    last_car = cumulative_abnormal_returns.iloc[-1]

    results.append({
        'security_ticker': company_ticker,
        'original_event_date': row['event_date'],
        'adjusted_event_date': event_date,
        'date': last_date,
        'CAR': last_car
    })

# Convert the final results to a DataFrame
results_df = pd.DataFrame(results)

# Check the first few rows of the results
print(results_df.head())


In [ ]:
!pip install openpyxl

In [ ]:
from google.colab import files

# Assuming you've already saved the file to 'event_study_results.xlsx'
results_df.to_excel('strong_meet_cur.xlsx', index=False)

# Download the file to your local machine
files.download('strong_meet_cur.xlsx')

In [ ]:
# import matplotlib.pyplot as plt

# results = []

# # Loop through each event
# for index, row in event_df.iterrows():
#     company_ticker = row['security_ticker']
#     event_date = row['event_date']

#     # Ensure event_date exists in the dataset, skip if it doesn't
#     if event_date not in df_clean.index:
#         print(f"Event date {event_date} for {company_ticker} not found in dataset. Skipping...")
#         continue

#     # Extract stock and market returns
#     company_returns = df_clean[company_ticker]
#     market_returns = df_clean['MOEX']

#     # Get the windows (estimation, event)
#     estimation_window, event_window = get_windows(df_clean, event_date, estimation_window_length, lag_period, event_window_length)

#     # Perform regression in the estimation window (Market Model)
#     X = sm.add_constant(estimation_window['MOEX'])
#     model = sm.OLS(estimation_window[company_ticker], X).fit()

#     # Get alpha and beta values
#     alpha, beta = model.params

#     # Calculate predicted returns and abnormal returns (AR) during the event window
#     predicted_returns = alpha + beta * event_window['MOEX']
#     abnormal_returns = event_window[company_ticker] - predicted_returns

#     # Calculate cumulative abnormal returns (CAR)
#     cumulative_abnormal_returns = abnormal_returns.cumsum()

#     # Store the results
#     for date, ar, car in zip(event_window.index, abnormal_returns, cumulative_abnormal_returns):
#         results.append({
#             'security_ticker': company_ticker,
#             'event_date': event_date,
#             'date': date,
#             'AR': ar,
#             'CAR': car
#         })

#     # Plot the CAR for this event
#     plt.figure(figsize=(10, 6))
#     plt.plot(event_window.index, cumulative_abnormal_returns, marker='o', linestyle='-', color='b')
#     plt.axvline(x=event_date, color='r', linestyle='--', label='Event Date')
#     plt.title(f'Cumulative Abnormal Returns (CAR) for {company_ticker} around {event_date}')
#     plt.xlabel('Date')
#     plt.ylabel('Cumulative Abnormal Return (CAR)')
#     plt.grid(True)
#     plt.xticks(rotation=45)
#     plt.legend()

#     # Save the plot as a PNG file
#     plt.savefig(f'CAR_plot_{company_ticker}_{event_date}.png')

#     # Show the plot in the notebook (optional, can be commented out if not needed)
#     plt.show()

# # Convert the results to a DataFrame
# results_df = pd.DataFrame(results)

# # Save to Excel
# results_df.to_excel('event_study_with_lag_results.xlsx', index=False)

# # Download the Excel file if running in Google Colab
# from google.colab import files
# files.download('event_study_with_lag_results.xlsx')